# 02 — Teste de inferência

Confere que o caminho completo funciona de ponta a ponta, numa amostra pequena:

**ler o dataset → montar o prompt congelado → gerar com o modelo → extrair a
letra → medir acurácia → gravar o JSONL.**

Não é um experimento. É o teste de fumaça que precisa passar antes de rodar o
baseline nos cinco datasets. Se algo aqui quebrar, quebra igual em escala, só
depois de horas de GPU.

Pré-requisitos:

```bash
python scripts/setup_datasets.py
python scripts/setup_models.py --models phi4-mini
# execute 01_formatacao_e_selecao.ipynb
```

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # pacote rmcq
sys.path.insert(0, str(ROOT / "scripts"))  # shims de compatibilidade

from rmcq.common import (
    build_answer_prompt,
    build_reflection_prompt,
    extract_final_answer,
    read_jsonl,
    write_jsonl,
)
from config import DATASETS, MODELS, RESULTS_DIR, SEED, SPLITS_DIR, STUDENT_GEN

pd.set_option("display.max_colwidth", 100)

# ---- parâmetros deste teste ----
DATASET = "arc"          # qualquer chave de config.DATASETS
SPLIT = "train"          # train (já selecionado por Cochran) ou test
MODEL = "phi4-mini"      # o menor dos quatro; troque para escalar o teste
N_QUESTIONS = 10
MAX_NEW_TOKENS = 640     # bem abaixo dos 4096 do experimento, só para ir rápido

print(f"dataset : {DATASET}/{SPLIT}  ({DATASETS[DATASET].problem_type})")
print(f"modelo  : {MODEL}  ->  {MODELS[MODEL].repo_id}  ({MODELS[MODEL].params})")
print(f"itens   : {N_QUESTIONS}")

dataset : arc/train  (knowledge)
modelo  : phi4-mini  ->  microsoft/Phi-4-mini-instruct  (3.8B)
itens   : 10


## 1. Leitura do dataset

Todo dataset é lido do mesmo jeito, do mesmo lugar, com os mesmos campos. Era
esse o ponto do notebook 01.

In [2]:
path = SPLITS_DIR / DATASET / f"{SPLIT}.jsonl"
if not path.exists():
    raise FileNotFoundError(f"{path} não existe. Execute 01_formatacao_e_selecao.ipynb primeiro.")

items = read_jsonl(path)
sample = items[:N_QUESTIONS]

print(f"{len(items):,} itens em {path.relative_to(ROOT)}")
pd.DataFrame([
    {
        "uid": i["uid"],
        "pergunta": i["question"][:70] + ("..." if len(i["question"]) > 70 else ""),
        "n_opções": i["num_choices"],
        "gabarito": i["answerKey"],
        "tem contexto": i["context"] is not None,
    }
    for i in sample
])

286 itens em data/splits/arc/train.jsonl


,uid,pergunta,n_opções,gabarito,tem contexto
0,arc-train-000002,A fold observed in layers of sedimentary rock most likely resulted fro...,4,B,False
1,arc-train-000003,Which of these do scientists offer as the most recent explanation as t...,4,D,False
2,arc-train-000008,Which of the following is a trait that a dog does NOT inherit from its...,4,C,False
3,arc-train-000013,The male insects in a population are treated to prevent sperm producti...,4,C,False
4,arc-train-000014,"On Earth, water can be a solid, a liquid, or a gas. Which energy sourc...",4,A,False
5,arc-train-000015,A ship leaks a large amount of oil near a coastal area. Which statemen...,4,B,False
6,arc-train-000017,"One evening as it is getting dark, Alex sits on the front porch and wa...",4,D,False
7,arc-train-000026,Stars are often classified by their apparent brightness in the nightti...,4,C,False
8,arc-train-000033,Lactose intolerance is a condition of the digestive system in which an...,4,A,False
9,arc-train-000034,Four materials are put into small containers. These materials are then...,4,A,False


## 2. O prompt, exatamente como o modelo vai receber

Vale olhar antes de gastar GPU. É o `ANSWER_PROMPT` de `scripts/common.py`, sem
nenhuma adaptação por modelo ou por dataset.

In [3]:
print(build_answer_prompt(sample[0]))
print("\n" + "=" * 78)
print(f"gabarito: {sample[0]['answerKey']}")

You are answering a multiple-choice question.

Question: A fold observed in layers of sedimentary rock most likely resulted from the

Options:
A) cooling of flowing magma.
B) converging of crustal plates.
C) deposition of river sediments.
D) solution of carbonate minerals.

Instructions:
- Think step by step before answering.
- Choose exactly one option.
- End your response with this exact line, and nothing after it:
FINAL ANSWER: <letter>

gabarito: B


### O extrator, antes do modelo

Se o extrator estiver errado, a acurácia do experimento está errada e nada no
resto do pipeline avisa. Os casos abaixo cobrem o formato exigido e as três
degradações que aparecem na prática: markdown em volta do rótulo, resposta em
prosa, e bloco `<think>` de modelos de raciocínio (Qwen3) contendo uma letra
diferente da conclusão.

In [4]:
extractor_cases = [
    ("formato exigido",        "Let me work through this.\n\nFINAL ANSWER: B",            "B"),
    ("markdown em volta",      "Reasoning here.\n\n**FINAL ANSWER:** (C)",                "C"),
    ("prosa",                  "After comparing them, the correct answer is D.",          "D"),
    ("letra solta",            "The reasoning points one way.\n\nA\n",                    "A"),
    ("<think> com outra letra", "<think>Maybe A works</think>\nActually no.\nFINAL ANSWER: C", "C"),
    ("repetido",               "FINAL ANSWER: A\nOn reflection, FINAL ANSWER: B",         "B"),
    ("abstenção",              "There is not enough information to decide.",              None),
]

rows = []
for name, text, expected in extractor_cases:
    ext = extract_final_answer(text, list("ABCD"))
    rows.append({
        "caso": name,
        "extraído": ext.letter,
        "esperado": expected,
        "método": ext.method,
        "seguiu formato": ext.followed_format,
        "ok": ext.letter == expected,
    })

extractor_df = pd.DataFrame(rows)
assert extractor_df["ok"].all(), "o extrator falhou em algum caso conhecido"
print("extrator: todos os casos passaram")
extractor_df

extrator: todos os casos passaram


,caso,extraído,esperado,método,seguiu formato,ok
0,formato exigido,B,B,strict,True,True
1,markdown em volta,C,C,loose_final,False,True
2,prosa,D,D,answer_is,False,True
3,letra solta,A,A,bare_letter,False,True
4,<think> com outra letra,C,C,strict,True,True
5,repetido,B,B,strict,True,True
6,abstenção,None,None,none,False,True


## 3. Carregamento do modelo

`ModelRunner` resolve dtype, `device_map`, `trust_remote_code` e chat template a
partir do `ModelSpec`. Um modelo de 8B em bfloat16 ocupa cerca de 16 GB de VRAM.

A primeira execução baixa os pesos se `setup_models.py` ainda não rodou.

In [5]:
import torch

print(f"torch {torch.__version__}   cuda disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i}  {p.name}  {p.total_memory / 1024**3:.1f} GiB")
else:
    print("  AVISO: sem GPU, um modelo de 8B na CPU leva minutos por questão")

torch 2.13.0+cu130   cuda disponível: True
  cuda:0  NVIDIA A100 80GB PCIe  79.3 GiB


In [6]:
from inference import ModelRunner, accuracy   # shim sobre rmcq.backends

runner = ModelRunner(MODEL)
print(f"carregado em: {runner.model.device}")
print(f"parâmetros  : {sum(p.numel() for p in runner.model.parameters()) / 1e9:.2f}B")

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
16:22:49 INFO    carregando phi4-mini (microsoft/Phi-4-mini-instruct) via transformers


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

16:22:54 INFO      device=cuda:0  max_len=131072


carregado em: cuda:0
parâmetros  : 3.84B


## 4. Uma questão, para inspecionar a saída bruta

Antes do laço, um item só, com a resposta completa impressa. É aqui que se vê se
o modelo raciocina passo a passo, se termina na linha exigida, e se há qualquer
coisa depois dela.

In [7]:
record = runner.answer(sample[0], stage="smoke", condition="no_reflection",
                       max_new_tokens=MAX_NEW_TOKENS)

print(record.raw_output)
print("=" * 78)
print(f"extraído {record.predicted}  |  gabarito {record.gold}  |  "
      f"{'CERTO' if record.is_correct else 'ERRADO'}")
print(f"método '{record.extraction_method}'  seguiu formato: {record.followed_format}")
print(f"{record.prompt_tokens} tokens de entrada, {record.completion_tokens} de saída, "
      f"{record.latency_s}s")

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

To determine the most likely cause of a fold observed in layers of sedimentary rock, let's analyze each option step by step:

A) Cooling of flowing magma: This process typically results in igneous rock, not sedimentary rock, and does not usually create folds in sedimentary layers.

B) Converging of crustal plates: This process is associated with the formation of mountain ranges and can cause folding in sedimentary rock layers, but it is more commonly linked to tectonic activity rather than the deposition of sediments.

C) Deposition of river sediments: This process involves the accumulation of sediments in layers, which can later be subjected to pressure and form folds. This is a common geological process that can lead to the formation of folds in sedimentary rock layers.

D) Solution of carbonate minerals: This process can lead to the dissolution of minerals and the formation of features like caves, but it does not typically result in the folding of sedimentary rock layers.

Based on 

## 5. O laço sobre a amostra

Sequencial e sem batching de propósito: o objetivo é validar corretude, não
throughput. O baseline de verdade vai precisar de batching ou vLLM.

In [8]:
from tqdm.auto import tqdm

records = []
for item in tqdm(sample, desc=f"{MODEL} em {DATASET}/{SPLIT}"):
    records.append(
        runner.answer(item, stage="smoke", condition="no_reflection",
                      max_new_tokens=MAX_NEW_TOKENS)
    )

results_df = pd.DataFrame([
    {
        "uid": r.uid,
        "pred": r.predicted,
        "gold": r.gold,
        "certo": r.is_correct,
        "método": r.extraction_method,
        "tokens_saída": r.completion_tokens,
        "latência_s": r.latency_s,
    }
    for r in records
])
results_df

phi4-mini em arc/train:   0%|          | 0/10 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

...

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

,uid,pred,gold,certo,método,tokens_saída,latência_s
0,arc-train-000002,C,B,False,strict,225,5.4522
1,arc-train-000003,D,D,True,strict,212,5.5424
2,arc-train-000008,C,C,True,strict,128,2.9456
3,arc-train-000013,C,C,True,strict,76,1.6513
4,arc-train-000014,A,A,True,strict,200,5.0809
5,arc-train-000015,B,B,True,strict,147,3.4516
6,arc-train-000017,D,D,True,strict,88,1.9829
7,arc-train-000026,C,C,True,strict,145,3.2123
8,arc-train-000033,A,A,True,strict,229,5.7961
9,arc-train-000034,A,A,True,strict,176,4.7704


## 6. Métricas

Três números que merecem leitura separada:

- **`accuracy_strict`** conta abstenção como erro. É o número que vai para o paper.
- **`accuracy_answered`** ignora abstenções. A diferença entre os dois isola
  incapacidade de resolver de incapacidade de seguir o formato.
- **`format_adherence`** é a fração que terminou exatamente em `FINAL ANSWER: X`.
  Se cair muito para um modelo, isso é resultado a reportar, não bug a corrigir
  no extrator.

In [9]:
metrics = accuracy(records)
for k, v in metrics.items():
    print(f"  {k:<26} {v}")

chance = sum(1 / i["num_choices"] for i in sample) / len(sample)
print(f"\n  {'acurácia do chute':<26} {chance:.4f}")
print(f"  {'acima do chute':<26} {metrics['accuracy_strict'] - chance:+.4f}")

  n                          10
  n_answered                 10
  n_abstained                0
  n_correct                  9
  accuracy                   0.9
  accuracy_answered          0.9
  format_adherence           1.0
  extr_strict                10
  extr_loose_final           0
  extr_answer_is             0
  extr_value_match           0
  extr_bare_letter           0
  extr_none                  0
  accuracy_strict_format_only 0.9
  mean_prompt_tokens         116.4
  mean_completion_tokens     162.6
  total_tokens               2790
  mean_latency_s             3.9886
  accuracy_strict            0.9

  acurácia do chute          0.2500
  acima do chute             +0.6500


### Erros e abstenções

Com 10 itens não há conclusão estatística nenhuma. O que se olha aqui é
qualitativo: o modelo errou porque raciocinou mal, ou porque a resposta certa
estava no texto e o extrator não pegou?

In [10]:
problems = [r for r in records if not r.is_correct]
print(f"{len(problems)} de {len(records)} para inspecionar\n")

for r in problems[:3]:
    print("=" * 78)
    print(f"{r.uid}   pred={r.predicted}  gold={r.gold}  método='{r.extraction_method}'")
    print("-" * 78)
    print(r.raw_output.strip()[-700:])
    print()

1 de 10 para inspecionar

arc-train-000002   pred=C  gold=B  método='strict'
------------------------------------------------------------------------------
 more commonly linked to tectonic activity rather than the deposition of sediments.

C) Deposition of river sediments: This process involves the accumulation of sediments in layers, which can later be subjected to pressure and form folds. This is a common geological process that can lead to the formation of folds in sedimentary rock layers.

D) Solution of carbonate minerals: This process can lead to the dissolution of minerals and the formation of features like caves, but it does not typically result in the folding of sedimentary rock layers.

Based on this analysis, the most likely cause of a fold observed in layers of sedimentary rock is the deposition of river sediments.

FINAL ANSWER: C



## 7. Reflexão do professor sobre uma resposta errada

Um passo à frente, só para confirmar que o prompt de reflexão renderiza e que o
professor obedece a restrição central: **não revelar a resposta correta.** Se ele
revelar, a etapa de avaliação vaza o gabarito e o experimento inteiro perde
sentido.

Aqui aluno e professor são o mesmo modelo, o que é a condição de
autorreflexão. Na grade completa serão todos os pares viáveis.

In [11]:
target = problems[0] if problems else records[0]
item = next(i for i in sample if i["uid"] == target.uid)

reflection = runner.reflect(
    item,
    previous_answer=target.raw_output,
    was_correct=bool(target.is_correct),
    depth="simple",
    perspective="teacher",
    max_new_tokens=400,
)

print(reflection.text.strip())
print("\n" + "=" * 78)
print(f"{len(reflection.text.split())} palavras, {reflection.completion_tokens} tokens, "
      f"{reflection.latency_s}s")

# Checagem de vazamento: a reflexão não deve nomear a alternativa correta.
gold_text = next(c["text"] for c in item["choices"] if c["label"] == item["answerKey"])
leaked_letter = extract_final_answer(reflection.text, [item["answerKey"]]).letter is not None
leaked_text = gold_text.lower() in reflection.text.lower()
print(f"vazou a letra do gabarito: {leaked_letter}")
print(f"vazou o texto do gabarito: {leaked_text}")

phi4-mini (hf):   0%|          | 0/1 [00:00<?, ?it/s]

The student model's reasoning was thorough, effectively analyzing each option and considering the geological processes involved. However, the primary source of error was in the analysis of option B, where the student overlooked that converging of crustal plates directly contributes to the folding of sedimentary layers due to tectonic forces. While the student correctly identified that deposition of river sediments (option C) can lead to folding, they incorrectly concluded it was the most likely cause. To improve, the student should carefully consider all factors and their direct impact on sedimentary rock formation, ensuring that the most direct and impactful processes are identified first. This case emphasizes the importance of distinguishing between processes that cause folding and those that primarily result in other geological features.

124 palavras, 144 tokens, 5.1239s
vazou a letra do gabarito: True
vazou o texto do gabarito: False


## 8. Gravação no formato único

Um JSONL, o mesmo `Record` que baseline, treino e avaliação vão usar. Colunas
que ainda não se aplicam ficam nulas em vez de ausentes, para que o arquivo
carregue direto num DataFrame sem alinhamento de schema.

In [12]:
out_path = RESULTS_DIR / "smoke" / f"{MODEL}_{DATASET}_{SPLIT}.jsonl"
n = write_jsonl(out_path, records)
print(f"{n} linhas -> {out_path.relative_to(ROOT)}")

reloaded = pd.DataFrame(read_jsonl(out_path))
print(f"recarregado: {reloaded.shape[0]} linhas x {reloaded.shape[1]} colunas")
print(f"\ncolunas: {list(reloaded.columns)}")
reloaded[["uid", "dataset", "student_model", "predicted", "gold", "is_correct",
          "extraction_method", "completion_tokens", "latency_s"]].head()

10 linhas -> results/smoke/phi4-mini_arc_train.jsonl
recarregado: 10 linhas x 28 colunas

colunas: ['uid', 'dataset', 'split', 'problem_type', 'stage', 'condition', 'student_model', 'teacher_model', 'prompt', 'raw_output', 'predicted', 'gold', 'is_correct', 'extraction_method', 'followed_format', 'reflection_depth', 'reflection_perspective', 'reflection_text', 'retrieved_uids', 'retrieved_similarities', 'k', 'prompt_tokens', 'completion_tokens', 'latency_s', 'seed', 'temperature', 'attempt', 'extra']


,uid,dataset,student_model,predicted,gold,is_correct,extraction_method,completion_tokens,latency_s
0,arc-train-000002,arc,phi4-mini,C,B,False,strict,225,5.4522
1,arc-train-000003,arc,phi4-mini,D,D,True,strict,212,5.5424
2,arc-train-000008,arc,phi4-mini,C,C,True,strict,128,2.9456
3,arc-train-000013,arc,phi4-mini,C,C,True,strict,76,1.6513
4,arc-train-000014,arc,phi4-mini,A,A,True,strict,200,5.0809


In [13]:
runner.unload()
if torch.cuda.is_available():
    print(f"VRAM alocada após liberar: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

16:24:28 INFO    liberando phi4-mini


VRAM alocada após liberar: 0.01 GiB


## O que este notebook não faz

Deliberadamente fora de escopo aqui, e necessário antes do baseline de verdade:

- **Batching.** Uma questão por vez desperdiça a GPU. `~1.800` questões de treino
  mais `~4.800` de teste, vezes quatro alunos, precisa de batching ou vLLM.
- **Retomada.** O baseline vai levar horas; precisa gravar incrementalmente e
  saber pular o que já fez, indexado por `uid`.
- **Troca de modelo em sequência.** `runner.unload()` existe para isso, mas o
  laço externo sobre os quatro modelos ainda não está escrito.
- **`max_new_tokens = 4096`.** Aqui usamos 640 para ir rápido. O experimento usa
  o valor do Caderno.

Rode este notebook uma vez com `MODEL = "phi4-mini"` e depois com cada um dos
outros três. Os pontos de atenção conhecidos: o Qwen3 tem modo de pensamento
híbrido (`QWEN_ENABLE_THINKING` em `config.py`) e o Llama 3 exige `HF_TOKEN`
com a licença aceita.